# Data Exploration — Air Quality & Low Emission Zone (LEZ) Paris

## Objective
First exploration of all data sources (Airparif NO₂, ZFE perimeters, INSEE IRIS income, weather)
to understand their structure and identify issues before cleaning and analysis.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from IPython.display import display

sys.path.insert(0, str(Path().resolve().parent))
from src.data_loader import load_airparif

print("Environment ready")


Environment ready


## 1. Airparif — NO₂ measurements

In [2]:
df_2019 = load_airparif(2019)
df_2020 = load_airparif(2020)
df_2021 = load_airparif(2021)
df_2022 = load_airparif(2022)
df_2023 = load_airparif(2023)

print("Data loaded ✓")

Data loaded ✓


In [3]:
for year, df in zip([2019, 2020, 2021, 2022, 2023], [df_2019, df_2020, df_2021, df_2022, df_2023]):
    print(f"{year}: {df.shape}")

2019: (8760, 41)
2020: (8784, 40)
2021: (8760, 40)
2022: (8760, 40)
2023: (8760, 40)


**Observations:**
- Datetime index parsed correctly
- 2019: 8,760 records, 41 stations
- 2020: 8,784 records (leap year), 40 stations
- 2021–2023: 8,760 records, 40 stations
- One station disappears between 2019 and 2020 — to identify and assess before analysis

### Identifying the missing station

In [4]:
stations_2019 = set(df_2019.columns)
stations_2020 = set(df_2020.columns)

disappeared = stations_2019 - stations_2020
appeared = stations_2020 - stations_2019

print(f"Station(s) present in 2019 but not in 2020: {disappeared}")
print(f"Station(s) present in 2020 but not in 2019: {appeared}")

Station(s) present in 2019 but not in 2020: {'PA04C'}
Station(s) present in 2020 but not in 2019: set()


**Observation — Station PA04C (Paris Centre):**
- Present only in 2019, absent from 2020 to 2023
- No replacement station identified
- Decision: exclude PA04C from the entire analysis to ensure temporal consistency

### Missing values

In [5]:
# Exclude PA04C upfront
dfs = {year: df.drop(columns=["PA04C"], errors="ignore")
       for year, df in zip([2019, 2020, 2021, 2022, 2023],
                           [df_2019, df_2020, df_2021, df_2022, df_2023])}

# Overall NaN rate per station (averaged across all 5 years)
nan_global = pd.concat(dfs.values()).isnull().mean() * 100
nan_global = nan_global.sort_values(ascending=False)

display(nan_global.round(1).to_frame("NaN %"))


,NaN %
DEF,38.1
BASCH,28.3
PA01H,22.7
SOULT,20.7
AUT,19.8
BONAP,8.9
EIFF3,8.8
STDEN,7.7
ARG,7.1
ELYS,6.4


In [6]:
# NaN rate per station and per year
nan_per_year = pd.DataFrame({
    year: df.isnull().mean() * 100
    for year, df in dfs.items()
})

display(nan_per_year.round(1))


,2019,2020,2021,2022,2023
A1,2.1,5.5,0.9,3.3,1.7
ARG,4.6,5.8,17.9,3.0,4.1
AUB,1.1,3.8,5.1,6.5,3.1
AUT,6.6,4.7,4.2,5.8,77.6
BASCH,100.0,11.1,4.6,4.7,21.1
BOB,2.9,15.5,0.7,6.9,0.9
BONAP,7.1,31.5,1.5,1.2,3.0
BP_EST,0.7,5.8,0.7,0.9,2.6
CELES,1.9,2.7,6.4,4.7,0.5
CHAMP,3.7,6.3,8.5,1.1,1.8


In [7]:
# Exclusion threshold: stations with more than 20% NaN over the full period
THRESHOLD = 20
excluded_stations = nan_global[nan_global > THRESHOLD].index.tolist()

print(f"Excluded stations (> {THRESHOLD}% NaN): {excluded_stations}")
print(f"Retained stations: {len(nan_global) - len(excluded_stations)} / {len(nan_global)}")

Excluded stations (> 20% NaN): ['DEF', 'BASCH', 'PA01H', 'SOULT']
Retained stations: 36 / 40


**Observations on Missing values:**
- 4 stations exceed the 20% NaN threshold and are excluded: DEF, BASCH, PA01H, SOULT
- The per-year breakdown reveals critical patterns:
  - DEF is 83.5% missing in 2021 and 100% in 2022 : absent during the core ZFE analysis window
  - BASCH is 100% missing in 2019 : no baseline data available
- AUT passes the global threshold (19.8%) but reaches 77.6% NaN in 2023 : flagged for monitoring
- Beyond 20%, imputation would introduce more noise than signal

- **36 stations retained out of 40**


### Outlier detection

In [8]:
EXCLUDED = ["PA04C", "DEF", "BASCH", "PA01H", "SOULT"]

clean_dfs = {year: df.drop(columns=EXCLUDED, errors="ignore")
             for year, df in zip([2019, 2020, 2021, 2022, 2023],
                               [df_2019, df_2020, df_2021, df_2022, df_2023])}

all_data = pd.concat(clean_dfs.values())

# Negative values (physically impossible for NO2)
neg = (all_data < 0).sum()
print("Negative values per station:")
neg_pos = neg[neg > 0]
if len(neg_pos):
    display(neg_pos.to_frame("Negative count"))
else:
    print("None")

# Extreme values
print("Max value per station:")
display(all_data.max().sort_values(ascending=False).round(1).to_frame("Max NO2 (µg/m³)"))


Negative values per station:


,Negative count
RUR-SO,19


Max value per station:


,Max NO2 (µg/m³)
RN20,245.8
AUT,233.9
OPERA,228.7
A1,223.7
CELES,213.7
BP_EST,213.7
RN2,211.0
HAUS,199.2
ELYS,182.1
GEN,180.2


**Observations on Outlier detection:**
- No systematic anomalies detected
- RUR-SO: 19 negative values — physically impossible, will be replaced by NaN in cleaning
- Max values (200–245 µg/m³) are physically plausible for high-traffic roadside stations during peak pollution episodes

In [9]:
# Datetime index consistency check
for year, df in clean_dfs.items():
    expected = pd.date_range(start=df.index.min(), end=df.index.max(), freq="h")
    missing_hours = expected.difference(df.index)
    duplicates = df.index.duplicated().sum()
    print(f"{year}: {len(missing_hours)} missing hours, {duplicates} duplicate timestamps")

2019: 0 missing hours, 0 duplicate timestamps
2020: 0 missing hours, 0 duplicate timestamps
2021: 0 missing hours, 0 duplicate timestamps
2022: 0 missing hours, 0 duplicate timestamps
2023: 0 missing hours, 0 duplicate timestamps


**Observations on Datetime index:**
- No missing hours and no duplicate timestamps across all 5 years
- The index is ready for time series analysis

## 2. ZFE perimeters

In [10]:
zfe = gpd.read_file("../data/raw/zfe_perimetres/zone-a-faibles-emissions.geojson")

print(f"Shape: {zfe.shape}")
print("Column dtypes:")
print(zfe.dtypes)
display(zfe.head())


Shape: (1, 17)
Column dtypes:
vp_horaires                             str
vul_horaires                            str
deux_rm_horaires                        str
id                                      str
date_debut                   datetime64[ms]
date_fin                             object
vp_critair                              str
vul_critair                             str
pl_critair                              str
pl_horaires                             str
autobus_autocars_critair                str
autobus_autocars_horaires               str
deux_rm_critair                         str
url_arrete                           object
url_site_information                    str
geo_point_2d                         object
geometry                           geometry
dtype: object


,vp_horaires,vul_horaires,deux_rm_horaires,id,date_debut,date_fin,vp_critair,vul_critair,pl_critair,pl_horaires,autobus_autocars_critair,autobus_autocars_horaires,deux_rm_critair,url_arrete,url_site_information,geo_point_2d,geometry
0,Mo-Fr 08:00-20:00; PH off,Mo-Fr 08:00-20:00; PH off,Mo-Fr 08:00-20:00; PH off,217500016-ZFE-001,2025-01-01,None,V3,V3,V3,Mo-Su 08:00-20:00,V3,Mo-Su 08:00-20:00,V3,None,https://www.zonefaiblesemissionsmetropolitaine...,"{'lon': 2.342916181995299, 'lat': 48.856615462...","POLYGON ((2.22422 48.85374, 2.22424 48.8542, 2..."


**Observations on ZFE perimeters:**
- Single perimeter covering the Paris metropolitan area
- Crit'Air 3 restriction applies to all vehicle types (vp_critair: V3)
- `date_debut` reflects the latest update (2025-01-01), not the original restriction date
- The Crit'Air 3 ban effective date (June 2021) will be hardcoded based on official sources
- `geometry` column contains the zone polygon — ready for spatial join with Airparif stations

## 3. INSEE IRIS — Median income

In [11]:
insee = pd.read_csv(
    "../data/raw/insee_iris/BASE_TD_FILO_IRIS_2021_DISP.csv",
    sep=";",
    dtype={"IRIS": str, "DEP": str}
)

print(f"Shape: {insee.shape}")
display(insee.head())


Shape: (16026, 30)


,IRIS,DISP_TP6021,DISP_INCERT21,DISP_Q121,DISP_MED21,DISP_Q321,DISP_EQ21,DISP_D121,DISP_D221,DISP_D321,...,DISP_PCHO21,DISP_PBEN21,DISP_PPEN21,DISP_PPAT21,DISP_PPSOC21,DISP_PPFAM21,DISP_PPMINI21,DISP_PPLOGT21,DISP_PIMPOT21,DISP_NOTE21
0,010040101,"19,0",2,14990,20350,26140,"0,55",11620,14280,16080,...,"3,0","3,6","26,9","6,2","8,6","3,3","3,8","1,5","-12,5",0
1,010040102,"25,0",1,13880,18570,24760,"0,59",10580,12890,14660,...,"4,2","2,4","24,9","5,8","11,1","3,7","5,1","2,3","-12,4",0
2,010040201,"19,0",1,15190,20700,27180,"0,58",11400,14060,16320,...,"3,5","4,0","27,2","6,4","7,7","2,8","3,3","1,6","-13,8",0
3,010040202,"8,0",1,19600,25230,33170,"0,54",14810,18310,20780,...,"2,4","3,6","23,8","16,2","4,0","1,8","1,5","0,7","-17,3",0
4,010330102,"24,0",1,14050,20420,29640,"0,76",9410,12570,15130,...,"4,9","1,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1",0


In [12]:
# Filter on Paris and inner suburbs (departments 75, 92, 93, 94)
idf_depts = ["75", "92", "93", "94"]
insee_idf = insee[insee["IRIS"].str[:2].isin(idf_depts)]
print(f"Île-de-France IRIS: {len(insee_idf)} / {len(insee)}")

Île-de-France IRIS: 2745 / 16026


**Observations on INSEE IRIS:**
- 16,026 IRIS nationally — filtered to 2,745 for Paris and inner suburbs (depts 75, 92, 93, 94)
- Key column: `DISP_MED21` (median disposable income)
- All numeric columns stored as strings with comma decimal separator — to be converted in cleaning
- Department code extracted from first 2 digits of IRIS identifier

## 4. Weather data (Open-Meteo)

In [13]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 48.8566,
    "longitude": 2.3522,
    "start_date": "2019-01-01",
    "end_date": "2023-12-31",
    "hourly": "windspeed_10m,precipitation,temperature_2m",
    "timezone": "Europe/Paris"
}

response = requests.get(url, params=params)
meteo_raw = response.json()

meteo = pd.DataFrame(meteo_raw["hourly"])
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.set_index("time")

print(f"Shape: {meteo.shape}")
print(f"NaN total: {meteo.isnull().sum().sum()}")
display(meteo.head())


Shape: (43824, 3)
NaN total: 0


,windspeed_10m,precipitation,temperature_2m
time,,,
2019-01-01 00:00:00,4.5,0.0,6.3
2019-01-01 01:00:00,6.2,0.0,6.6
2019-01-01 02:00:00,6.4,0.0,5.6
2019-01-01 03:00:00,4.1,0.0,6.4
2019-01-01 04:00:00,4.1,0.0,6.3


**Observations on Weather data (Open-Meteo):**
- 43,824 hourly records covering 2019–2023 (5 years × 8,760h, accounting for leap year 2020)
- 3 variables: wind speed (m/s), precipitation (mm), temperature (°C)
- No missing values
- Datetime index aligned with Airparif data — ready for merge in cleaning